In [0]:
from pyspark.sql.functions import col, round, when, count, sum as spark_sum

validation_df = spark.table("workspace.default.bronze_validation_report")
summary_df = spark.table("workspace.default.bronze_summary_report")
history_df = spark.table("workspace.default.bronze_quality_history")

quality_by_table = (
    validation_df
    .groupBy("table_name")
    .agg(
        count("*").alias("total_rules"),
        spark_sum(when(col("failure_percentage") > 0, 1).otherwise(0)).alias("failed_rules"),
        round(100 - (spark_sum(col("failure_percentage")) / count("*")), 2).alias("quality_score")
    )
)

quality_by_rule = (
    validation_df
    .groupBy("table_name", "rule")
    .agg(
        count("*").alias("rule_count"),
        round(spark_sum(col("failure_percentage")), 2).alias("total_failure_percentage")
    )
)

quality_by_table.write.mode("overwrite").saveAsTable("workspace.default.gold_quality_by_table")
quality_by_rule.write.mode("overwrite").saveAsTable("workspace.default.gold_quality_by_rule")

display(quality_by_table)